# 一.缺失值处理

## 1.导入所需的包

In [15]:
# 1 导入所需要包
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import f1_score,roc_auc_score,recall_score,classification_report,precision_score,confusion_matrix
from sklearn.model_selection import train_test_split,cross_val_score,GridSearchCV,StratifiedKFold
from lightgbm import LGBMClassifier
from lightgbm import early_stopping
import optuna
from streamlit import metric

## 2.检查缺失值

In [16]:
df=pd.read_csv(r'C:\Users\wangg\PycharmProjects\olist订单物流延误预测\数据集\data_processed\data_clean.csv')
print(df.isna().sum())

order_id                   0
customer_id                0
customer_unique_id         0
customer_region            0
seller_id                  0
seller_region              0
product_id                 0
product_category_high      0
volume                     1
weight                     1
process_time              13
is_delay                   0
distance                 488
freight_total              0
price_total                0
purchase_hour              0
purchase_dayofweek         0
purchase_month             0
is_same_region             0
dtype: int64


订单处理时间、体积、重量缺失值很少，直接删去对应的缺失列,距离列使用划分后的训练集中相同的商家地区和客户地区列的距离的平均值填充

In [17]:
df=df.dropna(subset=['process_time','volume','weight'])
# 划分特征和目标列
feature_name=['product_category_high','volume','weight','process_time','distance','freight_total','price_total','purchase_hour','purchase_dayofweek','purchase_month','customer_region','seller_region']
target_name='is_delay'
X=df[feature_name]
y=df[target_name].values
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.25,random_state=42,stratify=y)
print(X_train.shape,X_test.shape)
print(y_train.shape,y_test.shape)
print(y_train.mean(),y_test.mean())
print(X_train.isna().sum())
print(X_test.isna().sum())

(74070, 12) (24690, 12)
(74070,) (24690,)
0.0796543809909545 0.07966788173349534
product_category_high      0
volume                     0
weight                     0
process_time               0
distance                 358
freight_total              0
price_total                0
purchase_hour              0
purchase_dayofweek         0
purchase_month             0
customer_region            0
seller_region              0
dtype: int64
product_category_high      0
volume                     0
weight                     0
process_time               0
distance                 130
freight_total              0
price_total                0
purchase_hour              0
purchase_dayofweek         0
purchase_month             0
customer_region            0
seller_region              0
dtype: int64


In [5]:
# 计算卖家延期率
# seller_delay_rate = X_train.groupby('seller_id')['is_delay'].mean()
# global_rate = X_train['is_delay'].mean()
# X_train['seller_delay_rate'] = X_train['seller_id'].map(seller_delay_rate).fillna(global_rate)
# X_test['seller_delay_rate'] = X_test['seller_id'].map(seller_delay_rate).fillna(global_rate)
# 3. 删除 seller_id和is_delay列
# X_train = X_train.drop('seller_id', axis=1)
# X_train=X_train.drop('is_delay',axis=1)
# X_test = X_test.drop('seller_id', axis=1)
# X_test=X_test.drop('is_delay',axis=1)
print(X_train.columns.tolist())

['product_category_high', 'volume', 'weight', 'process_time', 'distance', 'freight_total', 'price_total', 'purchase_hour', 'purchase_dayofweek', 'purchase_month', 'customer_region', 'seller_region']


尝试：
用 seller_id 计算每个卖家的平均延期率
结果：交叉验证 AUC 0.7975，测试集 AUC 0.7029（严重过拟合），所以删去延期率特征

In [18]:
fill_values = X_train.groupby(['seller_region', 'customer_region'])['distance'].median()
# 4. 定义填充函数
def fill_distance(df, fill_values):
    df_filled = df.copy()
    for (sr, cr), median_dist in fill_values.items():
        mask = (df_filled['seller_region'] == sr) & (df_filled['customer_region'] == cr) & df_filled['distance'].isna()
        df_filled.loc[mask, 'distance'] = median_dist
    global_median = fill_values.median()
    df_filled['distance'] = df_filled['distance'].fillna(global_median)
    return df_filled
X_train = fill_distance(X_train, fill_values)
X_test = fill_distance(X_test, fill_values)
# 验证
print(X_train.isna().sum())
print(X_test.isna().sum())

product_category_high    0
volume                   0
weight                   0
process_time             0
distance                 0
freight_total            0
price_total              0
purchase_hour            0
purchase_dayofweek       0
purchase_month           0
customer_region          0
seller_region            0
dtype: int64
product_category_high    0
volume                   0
weight                   0
process_time             0
distance                 0
freight_total            0
price_total              0
purchase_hour            0
purchase_dayofweek       0
purchase_month           0
customer_region          0
seller_region            0
dtype: int64


# 二.异常值检查与处理

In [19]:
# 看极端值的延误率是否与整体不同
for col in ['price_total', 'freight_total', 'distance']:
    q99 = df[col].quantile(0.99)
    extreme = df[df[col] > q99]
    normal = df[df[col] <= q99]
    print(f"{col}: 极端值延误率 {extreme['is_delay'].mean():.2%}, "
          f"正常 {normal['is_delay'].mean():.2%}")

price_total: 极端值延误率 9.33%, 正常 7.95%
freight_total: 极端值延误率 10.02%, 正常 7.95%
distance: 极端值延误率 11.80%, 正常 7.92%


极端数据与延误率直接存在相关性，选择保留极端数据

# 三.基础模型构建

## 1.特征数据类型转换

In [24]:
# 1。特征数据类型转换
object_feature=['product_category_high','customer_region','seller_region']
for col in object_feature:
    X_train[col]=X_train[col].astype('category')
    X_test[col]=X_test[col].astype('category')

## 2.基础模型

In [46]:
lgbm=LGBMClassifier(n_estimators=1000,
                    learning_rate=0.02,
                    num_leaves=31,
                    max_depth=3,
                    min_child_samples=50,
                    subsample=0.8,
                    colsample_bytree=0.8,
                    reg_alpha=0.2,
                    reg_lambda=1,
                    random_state=42,
                    metric='auc',
                    objective='binary',
                    n_jobs=-1,
                    verbose=-1,
                    class_weight='balanced'
                    )

## 3.五折交叉验证

In [47]:
cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
scores=cross_val_score(lgbm,X_train,y_train,cv=cv,scoring='roc_auc')
print(f'五折交叉验证auc得分：{scores}')
print(f'五折交叉验证auc得分：{np.mean(scores):.4f}±{np.std(scores):.4f}')

五折交叉验证auc得分：[0.73645824 0.73239313 0.71656402 0.73280035 0.7264681 ]
五折交叉验证auc得分：0.7289±0.0070


## 4.optuna参数调优

In [49]:
# 简单验证一下是否过拟合
lgbm.fit(X_train,y_train)
print("测试集auc得分")
print(roc_auc_score(y_test,lgbm.predict_proba(X_test)[:,1]))

测试集auc得分
0.732041050255323


显示并不存在过拟合问题，因此不存在严重的信息泄露问题

In [51]:
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 500, 1000),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1),
        'num_leaves': trial.suggest_int('num_leaves', 20, 50),
        'max_depth': trial.suggest_int('max_depth', 3, 5),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 50),
        'subsample': trial.suggest_float('subsample', 0.6, 0.9),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 0.9),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.1, 1.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.1, 1.0),
    }
    lgbm = LGBMClassifier(**params,
                         random_state=42,
                         metric='auc',
                         objective='binary',
                         n_jobs=-1,
                         verbose=-1,
                         class_weight='balanced'
                         )
    cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
    auc_scores=cross_val_score(lgbm,X_train,y_train,cv=cv,scoring='roc_auc')
    return np.mean(auc_scores)
study = optuna.create_study(direction='maximize', study_name='lgbm_study',pruner=optuna.pruners.MedianPruner())
study.optimize(objective, n_trials=100)
print(f'Best params: {study.best_params}')
print(f'Best auc: {study.best_value:.4f}')

[I 2026-04-12 11:13:19,363] A new study created in memory with name: lgbm_study
[I 2026-04-12 11:13:24,197] Trial 0 finished with value: 0.7279153375285615 and parameters: {'n_estimators': 943, 'learning_rate': 0.05683169465354652, 'num_leaves': 33, 'max_depth': 4, 'min_child_samples': 41, 'subsample': 0.8348420176969423, 'colsample_bytree': 0.7102480941704834, 'reg_alpha': 0.18713188297243266, 'reg_lambda': 0.7417996914636442}. Best is trial 0 with value: 0.7279153375285615.
[I 2026-04-12 11:13:27,492] Trial 1 finished with value: 0.7311353719390458 and parameters: {'n_estimators': 784, 'learning_rate': 0.0392684127006221, 'num_leaves': 21, 'max_depth': 3, 'min_child_samples': 31, 'subsample': 0.7354930765490967, 'colsample_bytree': 0.7375081594684183, 'reg_alpha': 0.3763229207920292, 'reg_lambda': 0.2379216578406965}. Best is trial 1 with value: 0.7311353719390458.
[I 2026-04-12 11:13:32,613] Trial 2 finished with value: 0.7310602730461981 and parameters: {'n_estimators': 929, 'learn

Best params: {'n_estimators': 663, 'learning_rate': 0.027900804920016974, 'num_leaves': 23, 'max_depth': 5, 'min_child_samples': 48, 'subsample': 0.6928878957663047, 'colsample_bytree': 0.620589531626574, 'reg_alpha': 0.45432787975944483, 'reg_lambda': 0.26029212669761953}
Best auc: 0.7352


In [52]:
best_model=LGBMClassifier(**study.best_params,
                         random_state=42,
                         metric='auc',
                         objective='binary',
                         n_jobs=-1,
                         verbose=-1,
                         class_weight='balanced'
                         )
best_model.fit(X_train,y_train)
y_proba=best_model.predict_proba(X_test)[:,1]
print('测试集AUC:',roc_auc_score(y_test,y_proba))

测试集AUC: 0.7384201244577244


## 5.特征重要性分析

In [64]:
# 计算特征重要性
feature_importance = best_model.booster_.feature_importance(importance_type='gain')
feature_names = best_model.feature_name_
importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': feature_importance
}).sort_values('importance', ascending=False)
importance_df['importance'] = importance_df['importance'] / importance_df['importance'].sum()
print('='*20+"基于信息增益的特征重要性"+"="*20)
print(importance_df)

====================基于信息增益的特征重要性====================
                  feature  importance
9          purchase_month    0.332382
4                distance    0.139502
5           freight_total    0.094902
3            process_time    0.081228
1                  volume    0.066331
6             price_total    0.058484
2                  weight    0.053141
10        customer_region    0.049315
0   product_category_high    0.048008
11          seller_region    0.028211
8      purchase_dayofweek    0.025386
7           purchase_hour    0.023108


## 6.尝试使用早停

In [ ]:
params = {
    'n_estimators': 1000,  # 设大
    'learning_rate': 0.024,
    'num_leaves': 44,
    'max_depth': 5,
    'min_child_samples': 49,
    'subsample': 0.764,
    'colsample_bytree': 0.608,
    'reg_alpha': 0.683,
    'reg_lambda': 0.47,
    'random_state': 42,
    'metric': 'auc',
    'objective': 'binary',
    'n_jobs': -1,
    'verbose': -1,
    'class_weight': 'balanced'
}
# 单折测试
from sklearn.model_selection import train_test_split
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2)
model = LGBMClassifier(**params)
model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)],
          callbacks=[early_stopping(50)])
print(f"早停轮数: {model.best_iteration_}")
print(f"早停AUC: {model.best_score_['valid_0']['auc']:.4f}")

Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[603]	valid_0's auc: 0.7349
早停轮数: 603
早停AUC: 0.7349


根据结果，是否早停并不影响模型性能，因此两者随意选择，选择不早停

In [50]:
numeric_cols = X_train.select_dtypes(include=['number']).columns
for col in numeric_cols:
    auc = roc_auc_score(y_train, X_train[col])
    print(f"{col}: {auc:.4f}")

volume: 0.5171
weight: 0.5204
process_time: 0.5347
distance: 0.5640
freight_total: 0.5410
price_total: 0.5194
purchase_hour: 0.5118
purchase_dayofweek: 0.4844
purchase_month: 0.4656


尝试计算单特征auc，所有特征均在0.5附近徘徊，甚至有低于0.5的特征，单个特征几乎没有预测能力。最终模型能做到 0.7349，说明模型成功捕捉了特征之间的交互效应。

## 7.尝试使用xgboost模型观察效果

In [27]:
from xgboost import XGBClassifier
scale_pos_weight = len(y_train[y_train == 0]) / len(y_train[y_train == 1])
def objective_xgb(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 500, 1000),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1),
        'max_depth': trial.suggest_int('max_depth', 3, 5),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'subsample': trial.suggest_float('subsample', 0.6, 0.9),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 0.9),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.1, 1.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.1, 1.0),
        'gamma': trial.suggest_float('gamma', 0.1, 1.0)
    }
    xgb = XGBClassifier(**params,
                      random_state=42,
                      objective='binary:logistic',
                      n_jobs=-1,
                      verbose=-1,
                      verbosity=0,
                      enable_categorical=True,
                      scale_pos_weight=scale_pos_weight
                      )
    cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
    auc_scores=cross_val_score(xgb,X_train,y_train,cv=cv,scoring='roc_auc')
    return np.mean(auc_scores)
study = optuna.create_study(direction='maximize', study_name='xgb_study',pruner=optuna.pruners.MedianPruner())
study.optimize(objective_xgb, n_trials=100)
print(f'Best params: {study.best_params}')
print(f'Best auc: {study.best_value:.4f}')
best_model=XGBClassifier(**study.best_params,
                      random_state=42,
                      objective='binary:logistic',
                      n_jobs=-1,
                      verbose=-1,
                      verbosity=0,
                      enable_categorical=True,
                      scale_pos_weight=scale_pos_weight
                      )
best_model.fit(X_train,y_train)
y_proba=best_model.predict_proba(X_test)[:,1]
print('测试集AUC:',roc_auc_score(y_test,y_proba))

[I 2026-04-12 13:50:42,463] A new study created in memory with name: xgb_study
[I 2026-04-12 13:50:50,916] Trial 0 finished with value: 0.731492952563755 and parameters: {'n_estimators': 992, 'learning_rate': 0.03394135878408723, 'max_depth': 3, 'min_child_weight': 9, 'subsample': 0.7062551748965129, 'colsample_bytree': 0.8281539272101285, 'reg_alpha': 0.9893141291849243, 'reg_lambda': 0.936838625863262, 'gamma': 0.5960300765418719}. Best is trial 0 with value: 0.731492952563755.
[I 2026-04-12 13:50:56,961] Trial 1 finished with value: 0.7304493502037529 and parameters: {'n_estimators': 820, 'learning_rate': 0.05243406584796981, 'max_depth': 4, 'min_child_weight': 6, 'subsample': 0.7062071642407814, 'colsample_bytree': 0.660806292922243, 'reg_alpha': 0.5870585895324643, 'reg_lambda': 0.18613437380473868, 'gamma': 0.6280090223530351}. Best is trial 0 with value: 0.731492952563755.
[I 2026-04-12 13:51:02,764] Trial 2 finished with value: 0.731785174401981 and parameters: {'n_estimators':

Best params: {'n_estimators': 734, 'learning_rate': 0.022332693084988246, 'max_depth': 5, 'min_child_weight': 7, 'subsample': 0.8493086461361925, 'colsample_bytree': 0.782881778279424, 'reg_alpha': 0.5750744355346524, 'reg_lambda': 0.9413557865455232, 'gamma': 0.7941172827916654}
Best auc: 0.7364
测试集AUC: 0.7396983556141905


测试集AUC: 0.7396983556141905，改为xgboost之后测试集auc几乎无变化，说明模型效果不变，选择lightgbm模型，效率更高
